# 01 – Explore the Data

Before we build anything, we need to understand what we're working with.

**MovieLens 1M** is a classic benchmark dataset:
- 6,040 users who joined the platform in 2000
- 3,706 movies rated between 1995–2000
- 1,000,209 ratings (1–5 stars)

The first thing to notice: only ~4% of all possible user-movie pairs have a rating.
That's the **sparsity problem**, and it's why collaborative filtering is hard.

In [ ]:
import sys
sys.path.insert(0, "../src")  # Make rec_sys importable from the notebook

from rec_sys.data import MovieLensLoader

loader = MovieLensLoader("../data/ml-1m")
loader.load()
print("Data loaded.")

## Dataset Statistics

`loader.stats()` computes the core numbers we care about.
Sparsity is the most important: the fraction of user-movie pairs with NO rating.

In [ ]:
stats = loader.stats()
for key, val in stats.items():
    if isinstance(val, float):
        print(f"{key:35s}: {val:.2f}")
    else:
        print(f"{key:35s}: {val:,}")

## Rating Distribution

Are users generous or stingy? A skewed distribution matters for the ALS model,
which treats ratings as "confidence" in an implicit preference signal.

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

rating_counts = Counter(r.rating for r in loader.ratings)
ratings_sorted = sorted(rating_counts.items())

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([str(r) for r, _ in ratings_sorted], [c for _, c in ratings_sorted], color="steelblue")
ax.set_xlabel("Rating value")
ax.set_ylabel("Count")
ax.set_title("Rating Distribution (MovieLens 1M)")
plt.tight_layout()
plt.show()

print(f"Most common rating: {rating_counts.most_common(1)[0]}")

## Ratings per User

A few power users dominate. Most users have rated very few movies —
this is the "long tail" characteristic of recommendation datasets.

In [ ]:
from collections import Counter

ratings_per_user = Counter(r.user_id for r in loader.ratings)
counts = sorted(ratings_per_user.values())

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(counts, bins=50, color="coral", edgecolor="white")
ax.set_xlabel("Ratings per user")
ax.set_ylabel("Number of users")
ax.set_title("Ratings per User (log-scale tail visible)")
plt.tight_layout()
plt.show()

print(f"Min ratings/user: {min(counts)}")
print(f"Max ratings/user: {max(counts)}")
print(f"Median          : {counts[len(counts)//2]}")

## Ratings per Movie

The movie distribution is even more skewed: a small number of blockbusters
dominate while thousands of movies have very few ratings.

This creates the **cold-start problem for items** too — not just new users.

In [ ]:
from rec_sys.model.cold_start import PopularityFallback
from rec_sys.data.schema import MovieId

popularity = PopularityFallback.compute_popularity(loader.ratings, weighting="count")
top_10 = PopularityFallback.top_k_popular(popularity, k=10)

print("Top 10 most-rated movies:")
for rank, (mid, count) in enumerate(top_10, 1):
    movie = loader.get_movie(MovieId(int(mid)))
    print(f"  {rank:2d}. {movie.title:<50s} {int(count):>5,} ratings")

## Key Takeaways

1. **~96% sparse**: most user-movie pairs are unrated. This makes matrix factorization
   the right tool — it learns from the small fraction that *is* observed.

2. **Long tail in both users and movies**: a few items dominate; most are niche.
   A good system should surface the niche items for the right users.

3. **Average rating is ~3.6**: users tend to rate things they already like,
   introducing selection bias. The model learns from what's observed, not what's absent.

Next: `02_build_the_matrix.ipynb` — turning these ratings into a matrix.